# 04 - Feature Engineering
Builds the model-ready feature table: technical indicators, cross-asset context, macro features and news sentiment. Targets are shifted explicitly (`make_targets`) - the only place future information enters.

> Logic lives in `src/data/feature_engineering.py`.

## 1. Preprocess one symbol (clean + align + context + macro)

In [ ]:
from src.data.data_processor import build_interim, save_interim

SYMBOL = 'NVDA'
bundle = build_interim(SYMBOL, 'stocks')
save_interim(bundle, SYMBOL)
price, context, macro = bundle['price'], bundle['context'], bundle['macro']
print(f'{SYMBOL}: {len(price)} rows | {len(context)} context series | macro cols: {list(macro.columns)}')

## 2. Technical indicators

In [ ]:
from src.data.feature_engineering import technical_indicators

technicals = technical_indicators(price)
ind_cols = ['rsi_14','macd_hist','bb_pctb','atr_14','close_vs_sma_50','vol_z','realized_vol_21']
technicals[ind_cols].tail()

## 3. Cross-asset context features
Trailing returns and rolling correlations of indices/ETFs/commodities/FX vs the target.

In [ ]:
from src.data.feature_engineering import context_features

ctx_feats = context_features(price, context, SYMBOL)
print(f'{ctx_feats.shape[1]} context features')
ctx_feats.filter(like='corr60').tail()

## 4. Macro features (leakage-safe daily alignment)

In [ ]:
from src.data.feature_engineering import macro_features

macro_feats = macro_features(macro)
macro_feats[['fed_funds_rate','cpi_yoy','unemployment']].dropna().tail()

## 5. News sentiment (VADER; FinBERT upgrade path)

In [ ]:
from src.data.feature_engineering import daily_sentiment

sentiment = daily_sentiment(None, price.index, method='vader')
sentiment.tail()

## 6. Assemble the full feature table + shifted targets

In [ ]:
from src.data.feature_engineering import build_feature_table, make_targets, save_processed

table = make_targets(build_feature_table(SYMBOL))
path = save_processed(table, SYMBOL)
print(f'saved {path.name}: {table.shape[0]} rows x {table.shape[1]} cols')
table[['Close','next_day_return','direction']].tail()